# `quick-start.ipynb`

- `.env`에 TAVILY_API_KEY 추가 필요 (웹 서치 할 때 사용)

## Deep Agent
기존 Langchain Agent 보다 아래 능력을 포함
1. 계획 세우기
1. 파일 시스템 Tools
1. 하위 에이전트 다루는 능력

uv pip install deepagents tavily-python ipykernel python-dotenv langchain-openai

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from typing import Literal
from tavily import TavilyClient
from langchain.tools import tool

TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def internet_search(
        query: str,
        max_results: int = 5, # = 뒤에는 기본값
        topic: Literal['general', 'news', 'finance'] = 'general',
        include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic
    )
        
#tavily_client.search(
#    '이란 전쟁은 어떻게 될거고, 미국 선거에 어떤 영향을 줄까?',
#    max_result=3
#)

In [3]:
from deepagents import create_deep_agent

SYSTEM_PROMPT="""You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

deep_agent = create_deep_agent(
    model='openai:gpt-5-nano',
    tools=[internet_search],
    system_prompt=SYSTEM_PROMPT,
)

In [4]:
deep_agent.invoke({
    'messages': [
        {'role': 'user', 'content': '이란전쟁과 미국중간선거와 한국 환율에 대해 정리해줘'}
    ]
})

{'messages': [HumanMessage(content='이란전쟁과 미국중간선거와 한국 환율에 대해 정리해줘', additional_kwargs={}, response_metadata={}, id='ab444606-fdc1-4c2d-9cf6-1d25708d055a'),
  AIMessage(content=[{'id': 'rs_0e991526334db3800069af9bf610388194b4a3e756e31be87b', 'summary': [], 'type': 'reasoning'}, {'type': 'text', 'text': '다음 두 가지를 먼저 확인하고 진행하면 더 정확하고 매끄러운 정리(.)를 드릴 수 있습니다.\n\n질문\n- 기간: 어느 시점을 기준으로 요약할까요? 최근 6–12개월의 동향, 또는 2024년 말부터 현재까지의 흐름처럼 특정 기간이 있나요?\n- 이란전쟁의 범주: “이란전쟁”으로 특정 이슈를 말씀하신 건가요? 예를 들어 이란 핵협상/핵 개발 이슈, 이란과 미국 간 군사적 긴장 고조, 중동 지역에서의 교전 가능성 등 구체적으로 어떤 측면을 원하나요?\n- 한국 환율의 초점: 원-달러 환율의 일반 흐름(외환시장 동향)만 요약하길 원하나요, 아니면 한국의 정책(금리, 외환보유액, 개입 가능성)이나 주요 수입/산업에 대한 영향까지 포함한 심층 분석을 원하나요?\n- 깊이와 형식: 간단한 요약(핵심 포인트 5–8개)만 원하나요, 아니면 섹션별로 원인-현황-영향-전망까지 자세한 분석을 포함한 보고서 형식을 원하나요?\n\n제가 확인되면:\n- 최신 기사와 데이터로 연결된 구조화된 요약 보고서를 작성합니다.\n- 필요 시 서로 다른 시나리오(낙관/비관)도 포함하고, 한국 정책 시사점 및 투자/리스크 포인트도 제시하겠습니다.\n\n원하시면 지금 바로 대략적인 구조(개요/섹션 구성)와 간단한 초안 요약을 드린 뒤, 최신 정보를 반영해 상세 버전을 작성해도 됩니다. 어떤 방식으로 진행할지 알려주세요.', 'annotations': [], 'id'